In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT linux -> Test on linux
SMOTE Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED       = 42
TEST_PROJ  = "linux"
TRAIN_FILE  = "../../../../embedding/primevul/codet5/p_train_embedded.jsonl"
TEST_FILE   = "../../../../embedding/primevul/codet5/p_test_embedded.jsonl"
EMB_KEY    = "emb"
OUTPUT_DIR = "results/smote"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                         optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on SMOTE-oversampled source (all-except-linux)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (linux)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled linux
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (linux Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except linux -> Test on linux (SMOTE) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/5] Load embeddings
    # ----------------------------------------------------------
    print("\n[1/5] Loading embeddings...")

    X_tr, y_tr, proj_tr = load_jsonl(TRAIN_FILE)
    X_te, y_te, proj_te = load_jsonl(TEST_FILE)

    # Training pool: all projects in train file EXCEPT linux
    train_mask = proj_tr != TEST_PROJ
    train_emb  = X_tr[train_mask].astype(np.float32)
    train_lbl  = y_tr[train_mask].astype(np.int32)

    # Test set: only linux from test file
    test_mask = proj_te == TEST_PROJ
    test_emb  = X_te[test_mask].astype(np.float32)
    test_lbl  = y_te[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check TRAIN_FILE path and project names.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {TEST_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_tr[train_mask]))}")
    print(f"      Test set (linux)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/5] Normalize
    # ----------------------------------------------------------
    print("\n[2/5] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/5] Apply SMOTE to training pool only
    # ----------------------------------------------------------
    print("\n[3/5] Applying SMOTE...")
    smote                        = SMOTE(random_state=SEED)
    train_emb_sm, train_lbl_sm   = smote.fit_resample(train_emb, train_lbl)
    train_emb_sm                 = train_emb_sm.astype(np.float32)
    train_lbl_sm                 = train_lbl_sm.astype(np.int32)
    print(f"      Before SMOTE : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After SMOTE  : {train_emb_sm.shape}  "
          f"vuln={train_lbl_sm.sum()}  benign={int((train_lbl_sm == 0).sum())}")

    # ----------------------------------------------------------
    # [4/5] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/5] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_sm, train_lbl_sm,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_sm, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_sm == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_sm, train_lbl_sm, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/5] Metrics
    # ----------------------------------------------------------
    print("\n[5/5] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (linux Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"       : "smote",
        "train"            : "all_except_linux",
        "test"             : TEST_PROJ,
        "n_train_projects" : int(len(set(proj_tr[train_mask]))),
        "n_train_samples"  : int(len(train_lbl)),
        "n_train_smote"    : int(len(train_lbl_sm)),
        "n_test_samples"   : int(len(test_lbl)),
        "n_vuln_train"     : int(train_lbl.sum()),
        "n_vuln_train_smote": int(train_lbl_sm.sum()),
        "n_vuln_test"      : int(test_lbl.sum()),
        "accuracy"         : round(float(accuracy),  4),
        "precision"        : round(float(precision), 4),
        "recall"           : round(float(recall),    4),
        "f1"               : round(float(f1),        4),
        "auc"              : round(float(auc),        4),
        "g_mean"           : round(float(g_mean),    4),
        "pf"               : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "smote_linux.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_smote_linux.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()


=== PrimeVul | Train on ALL except linux -> Test on linux (SMOTE) ===
    Device: mps

[1/5] Loading embeddings...
      Training pool             : (132585, 256)  vuln=3860  benign=128725
      Projects in training pool : 630
      Test set (linux)          : (4140, 256)  vuln=70  benign=4070
      Training imbalance ratio  : 0.0291

[2/5] Normalizing embeddings...

[3/5] Applying SMOTE...
      Before SMOTE : (132585, 256)  vuln=3860  benign=128725
      After SMOTE  : (257450, 256)  vuln=128725  benign=128725

[4/5] Running ensemble iterations...


Ensemble:   0%|          | 0/3 [00:00<?, ?iter/s]


      [Iteration 1/3] Training neural network...


Iter 1 - Phase 1:   0%|          | 0/50 [00:00<?, ?it/s]

      Iter 1 - Phase 1 | Epoch 1/50 - val_loss: 0.4916
      Iter 1 - Phase 1 | Epoch 2/50 - val_loss: 0.4426
      Iter 1 - Phase 1 | Epoch 3/50 - val_loss: 0.4214
      Iter 1 - Phase 1 | Epoch 4/50 - val_loss: 0.4006
      Iter 1 - Phase 1 | Epoch 5/50 - val_loss: 0.3772
      Iter 1 - Phase 1 | Epoch 6/50 - val_loss: 0.3532
      Iter 1 - Phase 1 | Epoch 7/50 - val_loss: 0.3292
      Iter 1 - Phase 1 | Epoch 8/50 - val_loss: 0.3070
      Iter 1 - Phase 1 | Epoch 9/50 - val_loss: 0.2871
      Iter 1 - Phase 1 | Epoch 10/50 - val_loss: 0.2709
      Iter 1 - Phase 1 | Epoch 11/50 - val_loss: 0.2543
      Iter 1 - Phase 1 | Epoch 12/50 - val_loss: 0.2409
      Iter 1 - Phase 1 | Epoch 13/50 - val_loss: 0.2274
      Iter 1 - Phase 1 | Epoch 14/50 - val_loss: 0.2157
      Iter 1 - Phase 1 | Epoch 15/50 - val_loss: 0.2053
      Iter 1 - Phase 1 | Epoch 16/50 - val_loss: 0.1971
      Iter 1 - Phase 1 | Epoch 17/50 - val_loss: 0.1870
      Iter 1 - Phase 1 | Epoch 18/50 - val_loss: 0.1786
 

...

      Iter 2 - Phase 2 | Epoch 1/30 - val_loss: 0.0982
      Iter 2 - Phase 2 | Epoch 2/30 - val_loss: 0.0917
      Iter 2 - Phase 2 | Epoch 3/30 - val_loss: 0.0932
      Iter 2 - Phase 2 | Epoch 4/30 - val_loss: 0.0876
      Iter 2 - Phase 2 | Epoch 5/30 - val_loss: 0.0857
      Iter 2 - Phase 2 | Epoch 6/30 - val_loss: 0.0838
      Iter 2 - Phase 2 | Epoch 7/30 - val_loss: 0.0823
      Iter 2 - Phase 2 | Epoch 8/30 - val_loss: 0.0818
      Iter 2 - Phase 2 | Epoch 9/30 - val_loss: 0.0802
      Iter 2 - Phase 2 | Epoch 10/30 - val_loss: 0.0777
      Iter 2 - Phase 2 | Epoch 11/30 - val_loss: 0.0789
      Iter 2 - Phase 2 | Epoch 12/30 - val_loss: 0.0779
      Iter 2 - Phase 2 | Epoch 13/30 - val_loss: 0.0755
      Iter 2 - Phase 2 | Epoch 14/30 - val_loss: 0.0778
      Iter 2 - Phase 2 | Epoch 15/30 - val_loss: 0.0779
      Iter 2 - Phase 2 | Epoch 16/30 - val_loss: 0.0734
      Iter 2 - Phase 2 | Epoch 17/30 - val_loss: 0.0729
      Iter 2 - Phase 2 | Epoch 18/30 - val_loss: 0.0718
 

Iter 3 - Phase 1:   0%|          | 0/50 [00:00<?, ?it/s]

      Iter 3 - Phase 1 | Epoch 1/50 - val_loss: 0.4772
      Iter 3 - Phase 1 | Epoch 2/50 - val_loss: 0.4448
      Iter 3 - Phase 1 | Epoch 3/50 - val_loss: 0.4237
      Iter 3 - Phase 1 | Epoch 4/50 - val_loss: 0.4026
      Iter 3 - Phase 1 | Epoch 5/50 - val_loss: 0.3790
      Iter 3 - Phase 1 | Epoch 6/50 - val_loss: 0.3538
      Iter 3 - Phase 1 | Epoch 7/50 - val_loss: 0.3293
      Iter 3 - Phase 1 | Epoch 8/50 - val_loss: 0.3070
      Iter 3 - Phase 1 | Epoch 9/50 - val_loss: 0.2864
      Iter 3 - Phase 1 | Epoch 10/50 - val_loss: 0.2688
      Iter 3 - Phase 1 | Epoch 11/50 - val_loss: 0.2523
      Iter 3 - Phase 1 | Epoch 12/50 - val_loss: 0.2386
      Iter 3 - Phase 1 | Epoch 13/50 - val_loss: 0.2260
      Iter 3 - Phase 1 | Epoch 14/50 - val_loss: 0.2142
      Iter 3 - Phase 1 | Epoch 15/50 - val_loss: 0.2041
      Iter 3 - Phase 1 | Epoch 16/50 - val_loss: 0.1938
      Iter 3 - Phase 1 | Epoch 17/50 - val_loss: 0.1861
      Iter 3 - Phase 1 | Epoch 18/50 - val_loss: 0.1802
 

Iter 3 - Phase 2:   0%|          | 0/30 [00:00<?, ?it/s]

      Iter 3 - Phase 2 | Epoch 1/30 - val_loss: 0.0965
      Iter 3 - Phase 2 | Epoch 2/30 - val_loss: 0.0916


In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT tensorflow -> Test on tensorflow
SMOTE Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED       = 42
TEST_PROJ  = "tensorflow"
TRAIN_FILE  = "../../../../embedding/primevul/codet5/p_train_embedded.jsonl"
TEST_FILE   = "../../../../embedding/primevul/codet5/p_test_embedded.jsonl"
EMB_KEY    = "emb"
OUTPUT_DIR = "results/smote"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                         optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on SMOTE-oversampled source (all-except-tensorflow)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (tensorflow)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled tensorflow
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (tensorflow Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except tensorflow -> Test on tensorflow (SMOTE) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/5] Load embeddings
    # ----------------------------------------------------------
    print("\n[1/5] Loading embeddings...")

    X_tr, y_tr, proj_tr = load_jsonl(TRAIN_FILE)
    X_te, y_te, proj_te = load_jsonl(TEST_FILE)

    # Training pool: all projects in train file EXCEPT tensorflow
    train_mask = proj_tr != TEST_PROJ
    train_emb  = X_tr[train_mask].astype(np.float32)
    train_lbl  = y_tr[train_mask].astype(np.int32)

    # Test set: only tensorflow from test file
    test_mask = proj_te == TEST_PROJ
    test_emb  = X_te[test_mask].astype(np.float32)
    test_lbl  = y_te[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check TRAIN_FILE path and project names.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {TEST_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_tr[train_mask]))}")
    print(f"      Test set (tensorflow)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/5] Normalize
    # ----------------------------------------------------------
    print("\n[2/5] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/5] Apply SMOTE to training pool only
    # ----------------------------------------------------------
    print("\n[3/5] Applying SMOTE...")
    smote                        = SMOTE(random_state=SEED)
    train_emb_sm, train_lbl_sm   = smote.fit_resample(train_emb, train_lbl)
    train_emb_sm                 = train_emb_sm.astype(np.float32)
    train_lbl_sm                 = train_lbl_sm.astype(np.int32)
    print(f"      Before SMOTE : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After SMOTE  : {train_emb_sm.shape}  "
          f"vuln={train_lbl_sm.sum()}  benign={int((train_lbl_sm == 0).sum())}")

    # ----------------------------------------------------------
    # [4/5] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/5] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_sm, train_lbl_sm,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_sm, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_sm == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_sm, train_lbl_sm, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/5] Metrics
    # ----------------------------------------------------------
    print("\n[5/5] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (tensorflow Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"       : "smote",
        "train"            : "all_except_tensorflow",
        "test"             : TEST_PROJ,
        "n_train_projects" : int(len(set(proj_tr[train_mask]))),
        "n_train_samples"  : int(len(train_lbl)),
        "n_train_smote"    : int(len(train_lbl_sm)),
        "n_test_samples"   : int(len(test_lbl)),
        "n_vuln_train"     : int(train_lbl.sum()),
        "n_vuln_train_smote": int(train_lbl_sm.sum()),
        "n_vuln_test"      : int(test_lbl.sum()),
        "accuracy"         : round(float(accuracy),  4),
        "precision"        : round(float(precision), 4),
        "recall"           : round(float(recall),    4),
        "f1"               : round(float(f1),        4),
        "auc"              : round(float(auc),        4),
        "g_mean"           : round(float(g_mean),    4),
        "pf"               : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "smote_tensorflow.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_smote_tensorflow.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT vim -> Test on vim
SMOTE Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED       = 42
TEST_PROJ  = "vim"
TRAIN_FILE  = "../../../../embedding/primevul/codet5/p_train_embedded.jsonl"
TEST_FILE   = "../../../../embedding/primevul/codet5/p_test_embedded.jsonl"
EMB_KEY    = "emb"
OUTPUT_DIR = "results/smote"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                         optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on SMOTE-oversampled source (all-except-vim)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (vim)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled vim
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (vim Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except vim -> Test on vim (SMOTE) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/5] Load embeddings
    # ----------------------------------------------------------
    print("\n[1/5] Loading embeddings...")

    X_tr, y_tr, proj_tr = load_jsonl(TRAIN_FILE)
    X_te, y_te, proj_te = load_jsonl(TEST_FILE)

    # Training pool: all projects in train file EXCEPT vim
    train_mask = proj_tr != TEST_PROJ
    train_emb  = X_tr[train_mask].astype(np.float32)
    train_lbl  = y_tr[train_mask].astype(np.int32)

    # Test set: only vim from test file
    test_mask = proj_te == TEST_PROJ
    test_emb  = X_te[test_mask].astype(np.float32)
    test_lbl  = y_te[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check TRAIN_FILE path and project names.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {TEST_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_tr[train_mask]))}")
    print(f"      Test set (vim)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/5] Normalize
    # ----------------------------------------------------------
    print("\n[2/5] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/5] Apply SMOTE to training pool only
    # ----------------------------------------------------------
    print("\n[3/5] Applying SMOTE...")
    smote                        = SMOTE(random_state=SEED)
    train_emb_sm, train_lbl_sm   = smote.fit_resample(train_emb, train_lbl)
    train_emb_sm                 = train_emb_sm.astype(np.float32)
    train_lbl_sm                 = train_lbl_sm.astype(np.int32)
    print(f"      Before SMOTE : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After SMOTE  : {train_emb_sm.shape}  "
          f"vuln={train_lbl_sm.sum()}  benign={int((train_lbl_sm == 0).sum())}")

    # ----------------------------------------------------------
    # [4/5] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/5] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_sm, train_lbl_sm,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_sm, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_sm == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_sm, train_lbl_sm, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/5] Metrics
    # ----------------------------------------------------------
    print("\n[5/5] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (vim Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"       : "smote",
        "train"            : "all_except_vim",
        "test"             : TEST_PROJ,
        "n_train_projects" : int(len(set(proj_tr[train_mask]))),
        "n_train_samples"  : int(len(train_lbl)),
        "n_train_smote"    : int(len(train_lbl_sm)),
        "n_test_samples"   : int(len(test_lbl)),
        "n_vuln_train"     : int(train_lbl.sum()),
        "n_vuln_train_smote": int(train_lbl_sm.sum()),
        "n_vuln_test"      : int(test_lbl.sum()),
        "accuracy"         : round(float(accuracy),  4),
        "precision"        : round(float(precision), 4),
        "recall"           : round(float(recall),    4),
        "f1"               : round(float(f1),        4),
        "auc"              : round(float(auc),        4),
        "g_mean"           : round(float(g_mean),    4),
        "pf"               : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "smote_vim.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_smote_vim.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT gpac -> Test on gpac
SMOTE Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED       = 42
TEST_PROJ  = "gpac"
TRAIN_FILE  = "../../../../embedding/primevul/codet5/p_train_embedded.jsonl"
TEST_FILE   = "../../../../embedding/primevul/codet5/p_test_embedded.jsonl"
EMB_KEY    = "emb"
OUTPUT_DIR = "results/smote"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                         optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on SMOTE-oversampled source (all-except-gpac)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (gpac)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled gpac
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (gpac Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except gpac -> Test on gpac (SMOTE) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/5] Load embeddings
    # ----------------------------------------------------------
    print("\n[1/5] Loading embeddings...")

    X_tr, y_tr, proj_tr = load_jsonl(TRAIN_FILE)
    X_te, y_te, proj_te = load_jsonl(TEST_FILE)

    # Training pool: all projects in train file EXCEPT gpac
    train_mask = proj_tr != TEST_PROJ
    train_emb  = X_tr[train_mask].astype(np.float32)
    train_lbl  = y_tr[train_mask].astype(np.int32)

    # Test set: only gpac from test file
    test_mask = proj_te == TEST_PROJ
    test_emb  = X_te[test_mask].astype(np.float32)
    test_lbl  = y_te[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check TRAIN_FILE path and project names.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {TEST_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_tr[train_mask]))}")
    print(f"      Test set (gpac)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/5] Normalize
    # ----------------------------------------------------------
    print("\n[2/5] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/5] Apply SMOTE to training pool only
    # ----------------------------------------------------------
    print("\n[3/5] Applying SMOTE...")
    smote                        = SMOTE(random_state=SEED)
    train_emb_sm, train_lbl_sm   = smote.fit_resample(train_emb, train_lbl)
    train_emb_sm                 = train_emb_sm.astype(np.float32)
    train_lbl_sm                 = train_lbl_sm.astype(np.int32)
    print(f"      Before SMOTE : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After SMOTE  : {train_emb_sm.shape}  "
          f"vuln={train_lbl_sm.sum()}  benign={int((train_lbl_sm == 0).sum())}")

    # ----------------------------------------------------------
    # [4/5] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/5] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_sm, train_lbl_sm,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_sm, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_sm == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_sm, train_lbl_sm, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/5] Metrics
    # ----------------------------------------------------------
    print("\n[5/5] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (gpac Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"       : "smote",
        "train"            : "all_except_gpac",
        "test"             : TEST_PROJ,
        "n_train_projects" : int(len(set(proj_tr[train_mask]))),
        "n_train_samples"  : int(len(train_lbl)),
        "n_train_smote"    : int(len(train_lbl_sm)),
        "n_test_samples"   : int(len(test_lbl)),
        "n_vuln_train"     : int(train_lbl.sum()),
        "n_vuln_train_smote": int(train_lbl_sm.sum()),
        "n_vuln_test"      : int(test_lbl.sum()),
        "accuracy"         : round(float(accuracy),  4),
        "precision"        : round(float(precision), 4),
        "recall"           : round(float(recall),    4),
        "f1"               : round(float(f1),        4),
        "auc"              : round(float(auc),        4),
        "g_mean"           : round(float(g_mean),    4),
        "pf"               : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "smote_gpac.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_smote_gpac.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()

In [ ]:
"""
Cross-Project Vulnerability Detection on PrimeVul
--------------------------------------------------
Train on ALL projects EXCEPT FreeRDP -> Test on FreeRDP
SMOTE Oversampling
"""

import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, f1_score, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# Config
# =========================================================
SEED       = 42
TEST_PROJ  = "FreeRDP"
TRAIN_FILE  = "../../../../embedding/primevul/codet5/p_train_embedded.jsonl"
TEST_FILE   = "../../../../embedding/primevul/codet5/p_test_embedded.jsonl"
EMB_KEY    = "emb"
OUTPUT_DIR = "results/smote"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = "mps"
    torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = "cpu"


# =========================================================
# Data Loading
# =========================================================
def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    X        = np.array([r[EMB_KEY]   for r in records], dtype=np.float32)
    y        = np.array([r["target"]  for r in records], dtype=np.int32)
    projects = np.array([r["project"] for r in records], dtype=object)
    return X, y, projects


# =========================================================
# Neural Network
# =========================================================
class VulnerabilityClassifier(nn.Module):
    def __init__(self, input_dim):
        super(VulnerabilityClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# =========================================================
# Training Loop with Validation
# =========================================================
def train_neural_network(model, dataloader, x_val, y_val,
                         optimizer, criterion, epochs, desc):
    for epoch in tqdm(range(epochs), desc=desc, leave=False):
        model.train()
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val).squeeze(), y_val).item()
        print(f"      {desc} | Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f}")

    return model


# =========================================================
# Semi-supervised Transfer Learning
# =========================================================
def semi_supervised_transfer_learning(x_train, y_train, x_test, iteration):
    x_tr, x_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        random_state=SEED
    )

    x_tr_t   = torch.tensor(x_tr,   dtype=torch.float32)
    y_tr_t   = torch.tensor(y_tr,   dtype=torch.float32)
    x_val_t  = torch.tensor(x_val,  dtype=torch.float32).to(DEVICE)
    y_val_t  = torch.tensor(y_val,  dtype=torch.float32).to(DEVICE)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)

    dataset    = TensorDataset(x_tr_t, y_tr_t)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model     = VulnerabilityClassifier(x_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    criterion = nn.BCELoss()

    # Phase 1: Train on SMOTE-oversampled source (all-except-FreeRDP)
    model = train_neural_network(
        model, dataloader, x_val_t, y_val_t,
        optimizer, criterion,
        epochs=50,
        desc=f"Iter {iteration} - Phase 1"
    )

    # Pseudo-label the test set (FreeRDP)
    model.eval()
    with torch.no_grad():
        y_pred        = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()
        y_pred_binary = (y_pred > 0.5).astype(int)

    # Phase 2: Fine-tune on source + pseudo-labelled FreeRDP
    x_train_aug    = np.concatenate((x_train, x_test))
    y_train_aug    = np.concatenate((y_train, y_pred_binary))
    x_aug_t        = torch.tensor(x_train_aug, dtype=torch.float32)
    y_aug_t        = torch.tensor(y_train_aug, dtype=torch.float32)
    x_aug_val_t    = torch.tensor(x_val, dtype=torch.float32).to(DEVICE)
    y_aug_val_t    = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    dataset_aug    = TensorDataset(x_aug_t, y_aug_t)
    dataloader_aug = DataLoader(dataset_aug, batch_size=32, shuffle=True)

    model = train_neural_network(
        model, dataloader_aug, x_aug_val_t, y_aug_val_t,
        optimizer, criterion,
        epochs=30,
        desc=f"Iter {iteration} - Phase 2"
    )

    model.eval()
    with torch.no_grad():
        y_pred_final = model(x_test_t.to(DEVICE)).squeeze().cpu().numpy()

    return model, y_pred_final


# =========================================================
# XGBoost
# =========================================================
def train_base_model_xgb(x_train, y_train, sample_weights=None):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        eval_metric='logloss',
        random_state=SEED
    )
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


# =========================================================
# Confusion Matrix Plot
# =========================================================
def plot_confusion_matrix(tn, fp, fn, tp, save_path):
    cm = np.array([[tn, fp],
                   [fn, tp]])

    fig, ax = plt.subplots(figsize=(6, 5))
    img = ax.imshow(cm, interpolation='nearest')
    ax.set_title("Confusion Matrix (FreeRDP Test Set)")
    plt.colorbar(img, ax=ax)

    tick_marks = np.arange(2)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(["Non-Vulnerable", "Vulnerable"])
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(["Non-Vulnerable", "Vulnerable"])

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            color = "black" if cm[i, j] > thresh else "white"
            ax.text(j, i, cm[i, j], ha="center", va="center", color=color)

    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    print(f"Confusion matrix saved as {save_path}")
    plt.show()


# =========================================================
# Main
# =========================================================
def main():
    print(f"\n=== PrimeVul | Train on ALL except FreeRDP -> Test on FreeRDP (SMOTE) ===")
    print(f"    Device: {DEVICE}")

    # ----------------------------------------------------------
    # [1/5] Load embeddings
    # ----------------------------------------------------------
    print("\n[1/5] Loading embeddings...")

    X_tr, y_tr, proj_tr = load_jsonl(TRAIN_FILE)
    X_te, y_te, proj_te = load_jsonl(TEST_FILE)

    # Training pool: all projects in train file EXCEPT FreeRDP
    train_mask = proj_tr != TEST_PROJ
    train_emb  = X_tr[train_mask].astype(np.float32)
    train_lbl  = y_tr[train_mask].astype(np.int32)

    # Test set: only FreeRDP from test file
    test_mask = proj_te == TEST_PROJ
    test_emb  = X_te[test_mask].astype(np.float32)
    test_lbl  = y_te[test_mask].astype(np.int32)

    if train_emb.shape[0] == 0:
        raise ValueError("Training pool is empty. Check TRAIN_FILE path and project names.")
    if test_emb.shape[0] == 0:
        raise ValueError(f"No samples found for '{TEST_PROJ}' in {TEST_FILE}.")

    print(f"      Training pool             : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      Projects in training pool : {len(set(proj_tr[train_mask]))}")
    print(f"      Test set (FreeRDP)          : {test_emb.shape}  "
          f"vuln={test_lbl.sum()}  benign={int((test_lbl == 0).sum())}")
    print(f"      Training imbalance ratio  : {train_lbl.sum() / len(train_lbl):.4f}")

    # ----------------------------------------------------------
    # [2/5] Normalize
    # ----------------------------------------------------------
    print("\n[2/5] Normalizing embeddings...")
    scaler    = StandardScaler()
    train_emb = scaler.fit_transform(train_emb).astype(np.float32)
    test_emb  = scaler.transform(test_emb).astype(np.float32)

    # ----------------------------------------------------------
    # [3/5] Apply SMOTE to training pool only
    # ----------------------------------------------------------
    print("\n[3/5] Applying SMOTE...")
    smote                        = SMOTE(random_state=SEED)
    train_emb_sm, train_lbl_sm   = smote.fit_resample(train_emb, train_lbl)
    train_emb_sm                 = train_emb_sm.astype(np.float32)
    train_lbl_sm                 = train_lbl_sm.astype(np.int32)
    print(f"      Before SMOTE : {train_emb.shape}  "
          f"vuln={train_lbl.sum()}  benign={int((train_lbl == 0).sum())}")
    print(f"      After SMOTE  : {train_emb_sm.shape}  "
          f"vuln={train_lbl_sm.sum()}  benign={int((train_lbl_sm == 0).sum())}")

    # ----------------------------------------------------------
    # [4/5] Ensemble (NN + XGBoost) x 3 iterations
    # ----------------------------------------------------------
    print("\n[4/5] Running ensemble iterations...")
    num_iterations       = 3
    ensemble_predictions = np.zeros(len(test_lbl))

    for i in tqdm(range(num_iterations), desc="Ensemble", unit="iter"):
        print(f"\n      [Iteration {i+1}/{num_iterations}] Training neural network...")
        model_nn, _ = semi_supervised_transfer_learning(
            train_emb_sm, train_lbl_sm,
            test_emb,
            iteration=i+1
        )

        print(f"      [Iteration {i+1}/{num_iterations}] Training XGBoost...")
        model_nn.eval()
        with torch.no_grad():
            x_tr_t       = torch.tensor(train_emb_sm, dtype=torch.float32).to(DEVICE)
            y_train_pred = model_nn(x_tr_t).squeeze().cpu().numpy()

        sample_weights        = np.where(train_lbl_sm == 1, y_train_pred, 1 - y_train_pred)
        model_xgb             = train_base_model_xgb(train_emb_sm, train_lbl_sm, sample_weights)
        model_xgb_pred        = model_xgb.predict_proba(test_emb)[:, 1]
        ensemble_predictions += model_xgb_pred
        print(f"      [Iteration {i+1}/{num_iterations}] Done")

    # ----------------------------------------------------------
    # [5/5] Metrics
    # ----------------------------------------------------------
    print("\n[5/5] Computing metrics...")
    ensemble_avg = ensemble_predictions / num_iterations
    y_pred_final = (ensemble_avg > 0.5).astype(int)

    accuracy  = accuracy_score(test_lbl,  y_pred_final)
    recall    = recall_score(test_lbl,    y_pred_final, zero_division=0)
    precision = precision_score(test_lbl, y_pred_final, zero_division=0)
    auc       = roc_auc_score(test_lbl,   ensemble_avg)
    f1        = f1_score(test_lbl,        y_pred_final, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(test_lbl, y_pred_final).ravel()
    g_mean = np.sqrt((tp / (tp + fn + 1e-9)) * (tn / (tn + fp + 1e-9)))
    pf     = fp / (fp + tn + 1e-9)

    print("\n=== Evaluation Results (FreeRDP Test Set) ===")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")
    print(f"AUC       : {auc:.3f}")
    print(f"G-mean    : {g_mean:.3f}")
    print(f"PF value  : {pf:.3f}")
    print("\nConfusion Matrix:")
    print(f"  TN: {tn}  FP: {fp}")
    print(f"  FN: {fn}  TP: {tp}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    results = {
        "experiment"       : "smote",
        "train"            : "all_except_FreeRDP",
        "test"             : TEST_PROJ,
        "n_train_projects" : int(len(set(proj_tr[train_mask]))),
        "n_train_samples"  : int(len(train_lbl)),
        "n_train_smote"    : int(len(train_lbl_sm)),
        "n_test_samples"   : int(len(test_lbl)),
        "n_vuln_train"     : int(train_lbl.sum()),
        "n_vuln_train_smote": int(train_lbl_sm.sum()),
        "n_vuln_test"      : int(test_lbl.sum()),
        "accuracy"         : round(float(accuracy),  4),
        "precision"        : round(float(precision), 4),
        "recall"           : round(float(recall),    4),
        "f1"               : round(float(f1),        4),
        "auc"              : round(float(auc),        4),
        "g_mean"           : round(float(g_mean),    4),
        "pf"               : round(float(pf),        4),
        "confusion_matrix" : {
            "tn": int(tn), "fp": int(fp),
            "fn": int(fn), "tp": int(tp)
        }
    }

    json_path = os.path.join(OUTPUT_DIR, "smote_FreeRDP.json")
    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {json_path}")

    cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix_smote_FreeRDP.png")
    plot_confusion_matrix(tn, fp, fn, tp, cm_path)


if __name__ == "__main__":
    main()